In [ ]:
# Cell 1: Import Libraries
import pandas as pd
import json
import io
import requests
import os
from datetime import datetime

print("✓ All imports successful")
print(f"Starting analysis at: {datetime.now()}")

✓ All imports successful
Starting analysis at: 2025-10-09 11:03:04.108693


In [ ]:
# Cell 2: Define TCGA Collections to Process
tcga_collections = [
  'TCGA-LUAD',
  'TCGA-LUSC',
]

print("TCGA Collections to process:")
for i, collection in enumerate(tcga_collections, 1):
    print(f"  {i}. {collection}")

print(f"\nTotal collections: {len(tcga_collections)}")

TCGA Collections to process:
  1. TCGA-LUAD
  2. TCGA-LUSC

Total collections: 2


In [ ]:
# Cell 3: Test GDC Connection
print("Testing GDC connection...")

cases_endpt = 'https://api.gdc.cancer.gov/cases'

# Simple test query
test_filters = {
    "op": "in",
    "content": {
        "field": "project.project_id",
        "value": ["TCGA-LUAD"]  # Just test with one project
    }
}

test_params = {
    "filters": json.dumps(test_filters),
    "fields": "project.project_id,submitter_id",
    "format": "TSV",
    "size": "10"  # Just get 10 records for testing
}

try:
    response = requests.get(cases_endpt, params=test_params)
    if response.status_code == 200:
        print("✓ GDC connection successful")
        
        # Parse response
        output = response.content.decode('UTF-8')
        test_df = pd.read_csv(io.StringIO(output), sep='\t')
        print(f"✓ Retrieved {len(test_df)} test records")
        print("\nSample clinical data:")
        display(test_df.head())
    else:
        print(f"✗ GDC connection failed: HTTP {response.status_code}")
        
except Exception as e:
    print(f"✗ GDC connection error: {e}")

Testing GDC connection...
✓ GDC connection successful
✓ Retrieved 10 test records

Sample clinical data:


,id,project.project_id,submitter_id
0,70081320-540f-41d2-8687-ee6d011f8eb0,TCGA-LUAD,TCGA-MP-A4T9
1,397d3f69-1453-4057-b177-8723eec923d1,TCGA-LUAD,TCGA-97-8171
2,706420c4-8820-4b41-80d9-f3efd1d2a4f0,TCGA-LUAD,TCGA-MP-A4SV
3,3a23cdb5-2327-45ac-b0b5-d4afe038c757,TCGA-LUAD,TCGA-05-4430
4,3afa5045-a1ee-4a07-8aad-2e252b2f7d7a,TCGA-LUAD,TCGA-49-AAR0


In [ ]:
# Cell 4: Download Clinical Data from GDC
print("Downloading clinical data from GDC...")

cases_endpt = 'https://api.gdc.cancer.gov/cases'

filters = {
    "op": "in",
    "content": {
        "field": "project.project_id",
        "value": tcga_collections
    }
}

fields = [
    "project.project_id",
    "submitter_id",
    "primary_site",
    "diagnoses.primary_diagnosis",
    "diagnoses.treatments.treatment_or_therapy",
    "demographic.gender",
    "demographic.age_at_diagnosis",
]


params = {
    "filters": json.dumps(filters),
    "fields": ','.join(fields),
    "format": "TSV",
    "size": "10000",
    #"expand": "diagnoses, demographic, exposure, follow_up, pathology_detail, diagnoses.treatments"
}

try:
    print("Querying GDC API...")
    response = requests.get(cases_endpt, params=params)
    
    if response.status_code == 200:
        output = response.content.decode('UTF-8')
        clinical_data = pd.read_csv(io.StringIO(output), sep='\t')
        
        print(f"✓ Downloaded clinical data for {len(clinical_data)} patients")
        
        # Summary by project
        project_counts = clinical_data['project.project_id'].value_counts()
        print("\nClinical data by project:")
        for project, count in project_counts.items():
            print(f"  {project}: {count} patients")
            
    else:
        print(f"✗ Error downloading clinical data: HTTP {response.status_code}")
        clinical_data = pd.DataFrame()
        
except Exception as e:
    print(f"✗ Error: {e}")
    clinical_data = pd.DataFrame()

Querying GDC API...
✓ Downloaded clinical data for 1089 patients

Clinical data by project:
  TCGA-LUAD: 585 patients
  TCGA-LUSC: 504 patients


In [ ]:
# clinical_data is now a pandas DataFrame with clinical info which you can explore further. 
# If you would like to change the fields which are downloaded, then modify the 'fields' and "expand" lists above. I recommend checking the GDC documentation for all available fields.
# Here is the url for the data dictionary: https://docs.gdc.cancer.gov/Data_Dictionary/viewer/#?_top=1
# Here is the url for python examples for downloading the data: https://docs.gdc.cancer.gov/API/Users_Guide/Python_Examples/ 
# Here is an appendix for all of the fields available from the case endpoint: https://docs.gdc.cancer.gov/API/Users_Guide/Python_Examples/

In [ ]:
# Cell 6: Save Clinical Data
if not clinical_data.empty:
    clinical_data.to_csv('nsclc_clinical_data.csv', index=False)
    print("✓ Saved clinical data: nsclc_clinical_data.csv")

    # Create a clean version with non-empty columns only
    clean_clinical = clinical_data.dropna(axis=1, how='all')
    clean_clinical.to_csv('nsclc_clinical_data_clean.csv', index=False)
    print(f"✓ Saved clean clinical data: nsclc_clinical_data_clean.csv")
    print(f"  Original columns: {len(clinical_data.columns)}")
    print(f"  Clean columns: {len(clean_clinical.columns)}")
    
else:
    print("No clinical data to save")

✓ Saved clinical data: nsclc_clinical_data.csv
✓ Saved clean clinical data: nsclc_clinical_data_clean.csv
  Original columns: 201
  Clean columns: 177


In [ ]:
# Cell 7: Get RNA-seq File Inventory from GDC (Fixed Version)
print("Getting RNA-seq file inventory from GDC...")

files_endpt = 'https://api.gdc.cancer.gov/files'

# Based on diagnostic results, use correct field names and values
filters = {
    "op": "and",
    "content": [
        {
            "op": "in",
            "content": {
                "field": "cases.project.project_id",
                "value": tcga_collections
            }
        },
        {
            "op": "in", 
            "content": {
                "field": "data_category",
                "value": ["Transcriptome Profiling"]  # Correct capitalization
            }
        },
        {
            "op": "in",
            "content": {
                "field": "data_type",
                "value": ["Gene Expression Quantification"]  # Use data_type instead of workflow_type
            }
        },
        {
            "op": "in",
            "content": {
                "field": "experimental_strategy",
                "value": ["RNA-Seq"]
            }
        },
        {
            "op": "in",
            "content": {
                "field": "access",
                "value": ["open"]
            }
        },
        {
            "op": "in",
            "content": {
                "field": "cases.samples.sample_type",
                "value": ["Primary Tumor"]  # Correct sample type value
            }
        },
        {
            "op": "in",
            "content": {
                "field": "data_format",
                "value": ["TSV"]
            }
        }
    ]
}

params = {
    "filters": json.dumps(filters),
    "fields": "file_id,file_name,file_size,cases.submitter_id,cases.project.project_id,data_type,experimental_strategy,workflow_type",
    "format": "TSV",
    "size": "10000"
}

try:
    print("Querying GDC for RNA-seq files...")
    response = requests.get(files_endpt, params=params)
    
    if response.status_code == 200:
        output = response.content.decode('UTF-8')
        
        if output.strip():  # Check if response has content
            rnaseq_files = pd.read_csv(io.StringIO(output), sep='\t')
            
            print(f"✓ Found {len(rnaseq_files)} RNA-seq files")
            
            if not rnaseq_files.empty:
                # Show columns we actually got
                print(f"\nActual columns returned: {list(rnaseq_files.columns)}")
                
                # Summary by project (use correct column name)
                project_col = None
                for col in rnaseq_files.columns:
                    if 'project' in col.lower() and 'project_id' in col:
                        project_col = col
                        break
                
                if project_col:
                    project_counts = rnaseq_files[project_col].value_counts()
                    print(f"\nRNA-seq files by project:")
                    total_size_gb = 0
                    for project, count in project_counts.items():
                        project_files = rnaseq_files[rnaseq_files[project_col] == project]
                        size_gb = project_files['file_size'].sum() / (1024**3)
                        total_size_gb += size_gb
                        print(f"  {project}: {count} files ({size_gb:.1f} GB)")
                    
                    print(f"\nTotal download size: {total_size_gb:.1f} GB")
                
                # Show sample of files
                print(f"\nSample RNA-seq files:")
                display_cols = ['file_name', 'data_type'] + [col for col in rnaseq_files.columns if 'project' in col or 'submitter' in col]
                display_cols = [col for col in display_cols if col in rnaseq_files.columns]
                display(rnaseq_files[display_cols].head())
                
                # Check for workflow information
                if 'workflow_type' in rnaseq_files.columns:
                    workflow_types = rnaseq_files['workflow_type'].value_counts()
                    print(f"\nWorkflow types found:")
                    for workflow, count in workflow_types.items():
                        print(f"  {workflow}: {count} files")
                
            else:
                print("⚠ Query returned empty dataframe")
        else:
            print("✗ Empty response from GDC")
            rnaseq_files = pd.DataFrame()
    else:
        print(f"✗ Error getting RNA-seq files: HTTP {response.status_code}")
        print(f"Response text: {response.text[:500]}")
        rnaseq_files = pd.DataFrame()
        
except Exception as e:
    print(f"✗ Error: {e}")
    rnaseq_files = pd.DataFrame()

Getting RNA-seq file inventory from GDC...
Querying GDC for RNA-seq files...
✓ Found 1050 RNA-seq files

Actual columns returned: ['cases.0.project.project_id', 'cases.0.submitter_id', 'data_type', 'experimental_strategy', 'file_id', 'file_name', 'file_size', 'id']

RNA-seq files by project:
  TCGA-LUAD: 539 files (2.1 GB)
  TCGA-LUSC: 511 files (2.0 GB)

Total download size: 4.1 GB

Sample RNA-seq files:


,file_name,data_type,cases.0.project.project_id,cases.0.submitter_id
0,ee030015-242a-4dd1-b43c-2c97d5d365d9.rna_seq.a...,Gene Expression Quantification,TCGA-LUAD,TCGA-44-6147
1,9a5e0fa6-a785-4d8e-bca5-43e02a3965bf.rna_seq.a...,Gene Expression Quantification,TCGA-LUAD,TCGA-44-6147
2,f602ef77-0db5-47af-b259-e551c77a4281.rna_seq.a...,Gene Expression Quantification,TCGA-LUAD,TCGA-44-7661
3,59a0b90e-8b8b-41b8-95f0-d51961a94be5.rna_seq.a...,Gene Expression Quantification,TCGA-LUAD,TCGA-05-4396
4,9ebc079c-069f-4468-8b9e-680e39a3c4ad.rna_seq.a...,Gene Expression Quantification,TCGA-LUAD,TCGA-38-4629


In [ ]:
# Cell 8: Save RNA-seq File Inventory
if not rnaseq_files.empty:
    rnaseq_files.to_csv('nsclc_rnaseq_files_inventory.csv', index=False)
    print("✓ Saved RNA-seq inventory: nsclc_rnaseq_files_inventory.csv")
    
    # Create summary by project
    summary_stats = []
    
    # Find correct column names
    project_col = None
    patient_col = None
    
    for col in rnaseq_files.columns:
        if 'project' in col.lower() and 'project_id' in col:
            project_col = col
        if 'submitter' in col.lower():
            patient_col = col
    
    if project_col:
        for project in tcga_collections:
            project_files = rnaseq_files[rnaseq_files[project_col] == project]
            if not project_files.empty:
                unique_patients = project_files[patient_col].nunique() if patient_col else len(project_files)
                summary_stats.append({
                    'Project': project,
                    'Files': len(project_files),
                    'Unique_Patients': unique_patients,
                    'Size_GB': project_files['file_size'].sum() / (1024**3)
                })
    
    if summary_stats:
        rnaseq_summary = pd.DataFrame(summary_stats)
        rnaseq_summary.to_csv('nsclc_rnaseq_summary.csv', index=False)
        print("✓ Saved RNA-seq summary: nsclc_rnaseq_summary.csv")
        
        print(f"\nRNA-seq Summary:")
        display(rnaseq_summary)
    
    # Filter for STAR files if available
    if 'workflow_type' in rnaseq_files.columns:
        star_files = rnaseq_files[rnaseq_files['workflow_type'].str.contains('STAR', na=False)]
        if not star_files.empty:
            star_files.to_csv('nsclc_rnaseq_star_files.csv', index=False)
            print(f"✓ Saved STAR-specific files: nsclc_rnaseq_star_files.csv ({len(star_files)} files)")
    
else:
    print("No RNA-seq inventory to save")

✓ Saved RNA-seq inventory: nsclc_rnaseq_files_inventory.csv
✓ Saved RNA-seq summary: nsclc_rnaseq_summary.csv

RNA-seq Summary:


,Project,Files,Unique_Patients,Size_GB
0,TCGA-LUAD,539,516,2.127385
1,TCGA-LUSC,511,501,2.018934


In [ ]:
# Cell 9: Create Patient-Organized Download Scripts for NSCLC
print("Setting up variables for download script generation...")

# Load the RNA-seq files inventory
try:
    rnaseq_files = pd.read_csv('nsclc_rnaseq_files_inventory.csv')
    
    # Find RNA-seq column names
    rnaseq_patient_col = None
    project_col = None
    for col in rnaseq_files.columns:
        if 'submitter' in col.lower():
            rnaseq_patient_col = col
        if 'project' in col.lower() and 'project_id' in col:
            project_col = col
    
    total_rnaseq_gb = rnaseq_files['file_size'].sum() / (1024**3)
    
    print(f"✓ Variables loaded successfully:")
    print(f"  RNA-seq files: {len(rnaseq_files)} ({total_rnaseq_gb:.1f} GB)")
    
except Exception as e:
    print(f"✗ Error loading variables: {e}")
    print("Please make sure nsclc_rnaseq_files_inventory.csv exists.")
    rnaseq_files = pd.DataFrame()
    total_rnaseq_gb = 0
    rnaseq_patient_col = None
    project_col = None

# ===================================
# PATIENT-ORGANIZED RNA-SEQ SCRIPT
# ===================================

print(f"\n{'='*50}")
print("CREATING PATIENT-ORGANIZED RNA-SEQ SCRIPT")
print("="*50)

if not rnaseq_files.empty:
    rnaseq_organized_script = f'''#!/bin/bash
#
# NSCLC Patient-Organized RNA-seq Download
# Downloads RNA-seq files organized by: rna/PROJECT/PATIENT/
#

echo "NSCLC Patient-Organized RNA-seq Download Started: $(date)"
echo "=========================================="

# Create main RNA directory
mkdir -p rna_organized_nsclc
cd rna_organized_nsclc

echo "Organizing RNA-seq files by patient..."
echo "Total files: {len(rnaseq_files)}"
echo "Total size: {total_rnaseq_gb:.1f} GB"
echo ""

# Download counters
downloaded=0
failed=0
patients_processed=0

'''
    
    # Group RNA-seq files by project and patient
    if rnaseq_patient_col and project_col:
        rnaseq_by_patient = rnaseq_files.groupby([project_col, rnaseq_patient_col])
        
        total_patients_rna = len(rnaseq_files[rnaseq_patient_col].unique())
        
        for (project, patient_id), patient_files in rnaseq_by_patient:
            if pd.notna(project) and pd.notna(patient_id):
                
                rnaseq_organized_script += f'''
# Patient: {patient_id} ({project})
echo "Processing patient: {patient_id} ({project})"
mkdir -p {project}/{patient_id}

patients_processed=$((patients_processed + 1))
echo "  Patient $patients_processed/{total_patients_rna}: {patient_id}"
'''
                
                for _, file_row in patient_files.iterrows():
                    file_id = file_row['file_id']
                    file_name = file_row['file_name']
                    size_mb = file_row['file_size'] / (1024*1024)
                    
                    if pd.notna(file_id):
                        rnaseq_organized_script += f'''
echo "    Downloading {file_name} ({size_mb:.1f} MB)..."
if curl -f -L -o {project}/{patient_id}/{file_name} "https://api.gdc.cancer.gov/data/{file_id}"; then
    downloaded=$((downloaded + 1))
    echo "      ✓ Success"
else
    failed=$((failed + 1))
    echo "      ✗ Failed"
fi
'''
        
        rnaseq_organized_script += f'''
echo ""
echo "=========================================="
echo "Patient-organized RNA-seq download completed: $(date)"
echo "Patients processed: $patients_processed"
echo "Files downloaded successfully: $downloaded"
echo "Files failed: $failed"
echo "Data saved in: rna_organized_nsclc/"
echo "=========================================="
'''
    
    with open('bulk_download_nsclc_rnaseq.sh', 'w') as f:
        f.write(rnaseq_organized_script)
    os.chmod('bulk_download_nsclc_rnaseq.sh', 0o755)
    print(f"✓ Created: bulk_download_nsclc_rnaseq.sh")

print(f"\n📁 PATIENT-ORGANIZED SCRIPT CREATED:")
print(f"   bulk_download_nsclc_rnaseq.sh - RNA-seq by patient")

print(f"\n🗂️  RESULTING DIRECTORY STRUCTURE:")
print(f"   rna_organized_nsclc/")
print(f"     ├── TCGA-LUAD/")
print(f"     │   ├── TCGA-05-4244/")
print(f"     │   │   └── [RNA-seq TSV files]")
print(f"     │   └── TCGA-05-4249/")
print(f"     └── TCGA-LUSC/")
print(f"         ├── TCGA-18-3406/")
print(f"         │   └── [RNA-seq TSV files]")
print(f"         └── TCGA-18-3407/")

print(f"\n🚀 TO RUN DOWNLOAD:")
print(f"   nohup bash bulk_download_nsclc_rnaseq.sh > nsclc_download.log 2>&1 &")

Setting up variables for download script generation...
✓ Variables loaded successfully:
  RNA-seq files: 1050 (4.1 GB)

CREATING PATIENT-ORGANIZED RNA-SEQ SCRIPT
✓ Created: bulk_download_nsclc_rnaseq.sh

📁 PATIENT-ORGANIZED SCRIPT CREATED:
   bulk_download_nsclc_rnaseq.sh - RNA-seq by patient

🗂️  RESULTING DIRECTORY STRUCTURE:
   rna_organized_nsclc/
     ├── TCGA-LUAD/
     │   ├── TCGA-05-4244/
     │   │   └── [RNA-seq TSV files]
     │   └── TCGA-05-4249/
     └── TCGA-LUSC/
         ├── TCGA-18-3406/
         │   └── [RNA-seq TSV files]
         └── TCGA-18-3407/

🚀 TO RUN DOWNLOAD:
   nohup bash bulk_download_nsclc_rnaseq.sh > nsclc_download.log 2>&1 &
